# Multi-Model AI Detection — Sprint 3 Part 1: Text Model Fine-Tuning Pipeline

**Project ID:** `SKIT/AI/2023-2027/07`  
**Author:** Aryansh Agarwal (`23ESKCA021`) — Frontend Architecture & Text Detection Module  
**Model Architecture:** `microsoft/deberta-v3-base`  
**Dataset:** RAID Benchmark Split (`datasets/splits/text_manifest.csv` — 20,000 samples, 80/10/10 train/val/test)  
**Target Task:** Binary Sequence Classification (`0: human`, `1: ai`)  
**Target Environment:** Google Colab GPU runtime (NVIDIA T4 / V100 / A100)

## 1. Verify GPU Hardware
Ensure runtime type is set to **GPU** (Runtime -> Change runtime type -> T4 GPU).

In [ ]:
!nvidia-smi

## 2. Clone Repository & Setup Working Directory

In [ ]:
# Clone the repository
!git clone https://github.com/divishamb123/SKIT-AI-2023-2027-07.git
%cd SKIT-AI-2023-2027-07

## 3. Install Training Dependencies
Installs pinned training dependencies from `services/text-service/training/requirements-train.txt`.

In [ ]:
!pip install -r services/text-service/training/requirements-train.txt

## 4. Run Fine-Tuning Pipeline
Executes `finetune.py` with reproducible parameters:
- Model: `microsoft/deberta-v3-base`
- Epochs: 3
- Batch size: 8
- Learning rate: 2e-5 (AdamW)
- Max sequence length: 512
- Evaluates at each epoch and loads the best model checkpoint according to validation Macro F1 score.

In [ ]:
!python services/text-service/training/finetune.py \
    --model-name microsoft/deberta-v3-base \
    --output-dir services/text-service/outputs/deberta-v3-base-finetuned \
    --manifest-path datasets/splits/text_manifest.csv \
    --epochs 3 \
    --batch-size 8 \
    --learning-rate 2e-5 \
    --max-length 512

## 5. Verify Evaluation Metrics
Inspects `metrics.json` recorded during the evaluation pass.

In [ ]:
import json
from pathlib import Path

metrics_path = Path("services/text-service/outputs/deberta-v3-base-finetuned/metrics.json")
if metrics_path.exists():
    with open(metrics_path, "r", encoding="utf-8") as f:
        metrics = json.load(f)
    print(json.dumps(metrics, indent=2))
else:
    print("metrics.json not found. Check training output above.")

## 6. Export Fine-Tuned Model Weights
Creates a compressed zip archive of the model weights and tokenizer for deployment.

In [ ]:
!zip -r deberta_v3_finetuned.zip services/text-service/outputs/deberta-v3-base-finetuned

from google.colab import files
files.download("deberta_v3_finetuned.zip")

## 7. Serving Hand-Off Instructions
To deploy this fine-tuned checkpoint into the running platform:
1. Extract `deberta_v3_finetuned.zip` to `services/text-service/models/deberta-v3-base-finetuned`.
2. In `services/text-service/.env`, set:
   ```bash
   MODEL_PATH=models/deberta-v3-base-finetuned
   ```
3. Start or reload the text service (`docker-compose restart text-service` or `python services/text-service/app/main.py`).
4. The service will load the custom weights with explicit `id2label` mapping and serve inference via `POST /detect`.